# 01 注意力机制基础：从缩放点积到 MHA/MQA/GQA

## 简介

本笔记本介绍大语言模型最核心的组件——注意力机制（Attention）。
我们将从最基本的缩放点积注意力公式开始，逐步构建到多头注意力（MHA）、分组查询注意力（GQA）和多查询注意力（MQA），
并比较它们的参数量和 KV Cache 内存占用。

## 1. 导入

In [ ]:
import torch
import torch.nn.functional as F
from models.common.attention import MultiHeadAttention, GroupedQueryAttention

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

## 2. 缩放点积注意力（Scaled Dot-Product Attention）

注意力机制的核心公式：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

其中：
- **Q (Query)**: 查询向量，表示"我想查找什么"
- **K (Key)**: 键向量，表示"我能提供什么"
- **V (Value)**: 值向量，表示"我的实际内容"
- **√(d_k)**: 缩放因子，防止点积过大导致 softmax 梯度消失

让我们手动步骤步实现一次缩放点积注意力计算：

In [ ]:
# 步骤 1: 创建模拟的 Q, K, V 张量
B, S, D, n_heads = 2, 4, 64, 8
head_dim = D // n_heads  # 每个头的维度 = 8

# 模拟一个头的 Q, K, V
torch.manual_seed(42)
q = torch.randn(B, n_heads, S, head_dim)  # (2, 8, 4, 8)
k = torch.randn(B, n_heads, S, head_dim)
v = torch.randn(B, n_heads, S, head_dim)

print(f"Q 形状: {list(q.shape)}")
print(f"K 形状: {list(k.shape)}")
print(f"V 形状: {list(v.shape)}")

In [ ]:
# 步骤 2: 计算注意力得分 Q @ K^T / sqrt(d_k)
scale = head_dim ** 0.5  # sqrt(8) = 2.828
scores = (q @ k.transpose(-2, -1)) / scale  # (2, 8, 4, 4)
print(f"注意力得分形状: {list(scores.shape)}")
print(f"得分矩阵 (第一个 batch, 第一个头):")
print(scores[0, 0])

In [ ]:
# 步骤 3: 施加因果掩码 + Softmax
# 因果掩码：上三角为 True（将被mask为 -inf）
causal_mask = torch.triu(
    torch.ones(S, S, dtype=torch.bool), diagonal=1
).unsqueeze(0).unsqueeze(0)  # (1, 1, 4, 4)

print("因果掩码 (True=被屏蔽):")
for i in range(S):
    row = "".join("X" if causal_mask[0,0,i,j] else "O" for j in range(S))
    print(f"  token[{i}] -> {row}  (O=可见, X=被mask)")

scores_masked = scores.masked_fill(causal_mask, float('-inf'))
attn_weights = torch.softmax(scores_masked, dim=-1)
print(f"\n注意力权重形状: {list(attn_weights.shape)}")
print(f"权重矩阵(每行和=1):")
print(attn_weights[0, 0])

In [ ]:
# 步骤 4: 加权求和 attn_weights @ V
output = attn_weights @ v  # (2, 8, 4, 8)
print(f"输出形状: {list(output.shape)}")
print(f"第一个头的输出 (token 0): {output[0, 0, 0, :4]}")

## 3. 多头注意力 vs 分组查询 vs 多查询

| 类型 | Q 头数 | KV 头数 | 特点 |
|---|---|---|---|
| **MHA** | n_heads | n_heads | 每个 Q 头有独立的 KV，计算量最大|
| **GQA** | n_heads | n_kv_heads < n_heads | KV 头共享，平衡速度和质量|
| **MQA** | n_heads | 1 | 所有 Q 头共享 1 个 KV，KV Cache 最小|

我们的实现中，`GroupedQueryAttention` 当 `n_kv_heads==n_heads` 时等价于 MHA，当 `n_kv_heads==1` 时等价于 MQA。

In [ ]:
# 实例化三种注意力机制
dim, n_heads, n_kv_heads = 64, 8, 2

mha = MultiHeadAttention(dim=dim, n_heads=n_heads)
gqa = GroupedQueryAttention(dim=dim, n_heads=n_heads, n_kv_heads=n_kv_heads)
mqa = GroupedQueryAttention(dim=dim, n_heads=n_heads, n_kv_heads=1)

# 统计参数量
def count_params(model, name):
    n = sum(p.numel() for p in model.parameters())
    print(f"{name}: {n:,} 参数")
    return n

p_mha = count_params(mha, "MHA")
p_gqa = count_params(gqa, "GQA")
p_mqa = count_params(mqa, "MQA")
print(f"\nGQA 参数为 MHA 的 {p_gqa/p_mha*100:.1f}%")
print(f"MQA 参数为 MHA 的 {p_mqa/p_mha*100:.1f}%")

## 4. 实际运行验证

In [ ]:
# 前向传播并输出各步张量形状
x = torch.randn(2, 16, 64)  # (batch=2, seq=16, dim=64)

print("=" * 50)
print("MHA 前向传播：")
print("=" * 50)
mha = MultiHeadAttention(dim=64, n_heads=8)
out_mha = mha(x, use_causal_mask=True)
print(f"输入:  {list(x.shape)}")
print(f"输出: {list(out_mha.shape)}")
print(f"形状一致: {out_mha.shape == x.shape}")

print("\n" + "=" * 50)
print("MQA 前向传播 (n_kv_heads=1)：")
print("=" * 50)
mqa = GroupedQueryAttention(dim=64, n_heads=8, n_kv_heads=1)
out_mqa = mqa(x, use_causal_mask=True)
print(f"输入:  {list(x.shape)}")
print(f"输出: {list(out_mqa.shape)}")
print(f"形状一致: {out_mqa.shape == x.shape}")

## 5. KV Cache 内存占用对比

KV Cache 是推理时存储已计算的 Key 和 Value，避免重复计算。
其内存占用与 KV 头数成正比。

In [ ]:
def kv_cache_size(n_heads, n_kv_heads, head_dim, seq_len, batch_size=1, dtype_bytes=2):
    """计算 KV Cache 内存占用 (MB)"""
    # 每层需存 K + V
    k_bytes = batch_size * n_kv_heads * seq_len * head_dim * dtype_bytes
    v_bytes = batch_size * n_kv_heads * seq_len * head_dim * dtype_bytes
    return (k_bytes + v_bytes) / (1024 ** 2)

seq_len = 4096
head_dim = 64
batch_size = 1

# 对比 MHA(8头), GQA(8头Q,2头KV), MQA(8头Q,1头KV)
for n_q_heads, n_kv_heads, name in [(8, 8, "MHA"), (8, 2, "GQA"), (8, 1, "MQA")]:
    size_mb = kv_cache_size(n_q_heads, n_kv_heads, head_dim, seq_len, batch_size)
    print(f"{name}: KV heads={n_kv_heads}, Cache内存={size_mb:.2f} MB (每层)")

# 如果是 32 层的 LLaMA3-8B 级别模型
n_layers = 32
print(f"\n× {n_layers} 层总占用:")
for n_q_heads, n_kv_heads, name in [(8, 8, "MHA"), (8, 2, "GQA"), (8, 1, "MQA")]:
    size_mb = kv_cache_size(n_q_heads, n_kv_heads, head_dim, seq_len, batch_size) * n_layers
    print(f"  {name}: {size_mb:.0f} MB")

## 6. 注意力得分可视化

使用 ASCII 图展示注意力权重矩阵和因果掩码的效果。

In [ ]:
import math

# 模拟 8 个 token 的注意力分布
torch.manual_seed(0)
S_demo = 8
q_demo = torch.randn(1, 1, S_demo, 4)
k_demo = torch.randn(1, 1, S_demo, 4)
scores_demo = (q_demo @ k_demo.transpose(-2, -1)) / math.sqrt(4)
mask_demo = torch.triu(torch.ones(S_demo, S_demo, dtype=torch.bool), diagonal=1)
scores_masked_demo = scores_demo.masked_fill(mask_demo, float('-inf'))
attn_demo = torch.softmax(scores_masked_demo, dim=-1)[0, 0]

print("注意力权重热力图 (ASCII):")
print("  每行为一个 query token 对所有 key token 的注意力分布")
print("  '░'=低权重, '█'=高权重, '.'=mask掉(0)")
print()
chars = " .=os#@"
max_val = attn_demo.max().item()
print("   " + "".join(f" K{i}" for i in range(S_demo)))
for i in range(S_demo):
    row = f"Q{i} "
    for j in range(S_demo):
        if mask_demo[i, j]:
            row += " · "  # 被mask
        else:
            v = attn_demo[i, j].item() / max_val
            idx = min(int(v * (len(chars) - 1)), len(chars) - 1)
            row += f" {chars[idx]} "
    print(row)
print()
print("箭头方向:每个 Q token 只能看到自己和之前的 K/V token")

## 7. 关键要点总结

1. **缩放点积注意力**是所有注意力变体的基础，核心是 softmax(QK^T/√d_k) × V
2. **因果掩码**保证自回归生成时 token_i 只能看到 token_{0..i}
3. **MHA**: 每个 Q 头有独立的 KV，表达能力最强
4. **GQA**: KV 头数介于 1 和 n_heads 之间，LLaMA2/3 的默认选择
5. **MQA**: 只有 1 个 KV 头，KV Cache 最小，但质量略有下降
6. **KV Cache** 内存占用 = 2 × batch × n_kv_heads × seq_len × head_dim × dtype_bytes